# Circuit Obfuscation Using PyZX

We implement the algorithm described in [this blog post](https://blog.20squares.xyz/obfuscating-quantum-circuits-zx/) to perform *circuit obuscation*, i.e. prevent an attacker from retrieving a seed $r$ used to generate the circuit.

In [21]:
import pyzx as zx
from pyzx.circuit.gates import VertexType, EdgeType
import numpy as np
import random
from fractions import Fraction

qubit_amount = 10
depth = 120
# Only clifford+T gates
allowed_gates = ['H', 'S', 'CNOT', 'T']

We start by creating the random seed $r$ for circuit generation.

In [22]:
# Random seed of appropriate size that should be used for obfuscation
# size = int(np.log2(len(allowed_gates)) * 2 * np.log2(qubit_amount) * depth)
seed_gate = np.random.randint(0, len(allowed_gates), size=depth, dtype=np.uint32)
seed_qubit = np.random.randint(0, qubit_amount, size=2*depth, dtype=np.uint32)


In [23]:
from pyzx import gates
circuit = zx.Circuit(qubit_amount)

for i in range(depth):
    gate_type = allowed_gates[seed_gate[i]]
    
    # Extract candidate qubits
    q1 = seed_qubit[2*i]
    q2 = seed_qubit[2*i + 1]
    
    if gate_type == 'H':
        circuit.add_gate(gates.HAD(q1))
    elif gate_type == 'S':
        circuit.add_gate(gates.S(q1))
    elif gate_type == 'T':
        circuit.add_gate(gates.T(q1))
        
    elif gate_type == 'CNOT':
        # Logic check: A CNOT requires two different qubits
        if q1 != q2:
            circuit.add_gate(gates.CNOT(q1, q2))
        else:
            # Fallback or skip if qubits are the same
            circuit.add_gate(gates.HAD(q1)) 

print("Randomly generated circuit:")
zx.draw(circuit)

Randomly generated circuit:


We start by convert every CNOT gate to a Pauli gadget. This can be done by first converting the circuit to the ZX-diagram representation, and observing that since we work with a Clifford input circuit, the CNOT gates are simply two connected nodes on different qubits.

In [24]:
g = circuit.to_graph()

# Retrieve all CNOT edges
cnots = []
    
# Iterate through all edges in the graph
for edge in g.edges():
    u, v = g.edge_st(edge)

    if g.qubit(u) == g.qubit(v):
        continue  # Skip edges that connect vertices on the same qubit

    ctrl = u if g.type(u) == VertexType.Z else v
    targ = v if g.type(u) == VertexType.Z else u
    
    cnots.append((ctrl, targ))

Now that the CNOTs are identified, we can change them into Pauli gadgets.

In [25]:
g_gadgets = g.copy()

# Replace CNOTs with phase gadgets in the graph
for (ctrl, targ) in cnots:
    # Remove the original CNOT edge
    g_gadgets.remove_edge((ctrl, targ))
    
    # Change the phase (was 0)
    g_gadgets.set_phase(ctrl, Fraction(-1, 2))  # Z_c
    g_gadgets.set_phase(targ, Fraction(-1, 2))  # X_t

    hub, phase_vertex = g_gadgets.add_phase_gadget(Fraction(1, 2), targets=[targ, ctrl])

    # change edge to simple edge
    g_gadgets.remove_edge((targ, hub))
    g_gadgets.add_edge((targ, hub), EdgeType.SIMPLE)

g_gadgets.pack_circuit_rows()
zx.draw(g_gadgets)

Then, we want to use the fact that attaching two phase gadgets whose phases add up to zero to a set of Z-spiders simply is the identity operation. For this, we start by converting every X-spiders to Z-spiders by extracting the hadamards (i.e., flipping every edge to which the X-spider is attached).

In [26]:
g_green_only = g_gadgets.copy()

def opposite_edge_type(edge_type):
    if edge_type == EdgeType.SIMPLE:
        return EdgeType.HADAMARD
    elif edge_type == EdgeType.HADAMARD:
        return EdgeType.SIMPLE
    else:
        raise ValueError("Invalid edge type. Must be SIMPLE or HADAMARD.")
    
for v in g_green_only.vertices():
    if g_green_only.type(v) == VertexType.X:
        g_green_only.set_type(v, VertexType.Z)
        
        for e in g_green_only.incident_edges(v):
            g_green_only.set_edge_type(e, opposite_edge_type(g_green_only.edge_type(e)))

zx.draw(g_green_only)

We can therefore randomly place pairs of pauli gadgets.

In [27]:
INJECTION_NUMBER = 100
MAX_QUBIT_TARGETS = 10

g_randomized = g_green_only.copy()

# Only internal Z-spiders
valid_z_targets = [
    v for v in g_green_only.vertices()
    if g_green_only.type(v) == VertexType.Z
    and v not in g_green_only.inputs()
    and v not in g_green_only.outputs()
]

# Inject harmless noise to obfuscate
for _ in range(INJECTION_NUMBER):
    # Select random targets using pure Python lists (NO numpy arrays)
    num_targets = random.randint(2, min(MAX_QUBIT_TARGETS, len(valid_z_targets)))
    vertices_to_target = random.sample(valid_z_targets, num_targets)

    # Random angle using standard Python integers
    angle = Fraction(random.randint(0, 31), 16)  # angle * pi

    # --- Add +angle gadget ---
    hub1 = g_randomized.add_vertex(VertexType.Z, phase=0)
    phase1 = g_randomized.add_vertex(VertexType.Z, phase=angle)
    g_randomized.add_edge((hub1, phase1), EdgeType.HADAMARD)
    for target in vertices_to_target:
        g_randomized.add_edge((hub1, target), EdgeType.HADAMARD)

    # --- Add -angle gadget ---
    hub2 = g_randomized.add_vertex(VertexType.Z, phase=0)
    phase2 = g_randomized.add_vertex(VertexType.Z, phase=-angle)
    g_randomized.add_edge((hub2, phase2), EdgeType.HADAMARD)
    for target in vertices_to_target:
        g_randomized.add_edge((hub2, target), EdgeType.HADAMARD)


g_randomized.pack_circuit_rows()
zx.draw(g_randomized)

We check that the final tensor still represents the same linear map as the original one : if $G$ was the original diagram and $G'$ the obfuscated one, check that ${G'}^\dagger G = \mathbb{I}$.

In [28]:
g_randomized_inv = g_randomized.copy()
g_randomized_inv = g_randomized_inv.adjoint()

g_check = g_gadgets.copy()
g_check.compose(g_randomized_inv)

zx.full_reduce(g_check)

print("The final tensor remains unchanged :")
zx.compare_tensors(g_check, zx.Circuit(qubit_amount).to_graph(), preserve_scalar=False)

The final tensor remains unchanged :


True

As a bonus, we also notice that the standard methods for circuit extraction now give a different circuit when applied to the obfuscated version :

In [29]:
g_1 = g.copy()
g_2 = g_randomized.copy()

zx.full_reduce(g_1)
zx.full_reduce(g_2)

c1 = zx.extract_circuit(g_1)
c2 = zx.extract_circuit(g_2)

print("Extracted circuit from original graph:")
zx.draw(c1)
print("Extracted circuit from obfuscated graph:")
zx.draw(c2)


Extracted circuit from original graph:


Extracted circuit from obfuscated graph:


That is good news !

We save the two circuits for further study with quimb.

In [30]:
q1 = c1.to_qasm()
q2 = c2.to_qasm()

with open("../out/original_circuit.qasm", "w") as f:
    f.write(q1)
with open("../out/obfuscated_circuit.qasm", "w") as f:
    f.write(q2)